In [22]:
# Q10 - Best and worst segments

# Question: which combinations of customer and campaign features have the highest and lowest yes rate?

# A segment combines previous campaign outcome, contact type and age group, all known before a call is made (call length and the
# number of calls received are only known once a contact is under way, so they are not used). Each segment is compared with the yes
# rate of its own inferred year (decision 10), as "times the year's rate". A rate is shown only where a segment has at least 100
# customers in that year. A segment is more trustworthy if it is above (or below) its year's rate in every year where it can be
# measured.

In [23]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 23)

In [24]:
# Q10, step 1: segments by previous outcome, contact type and age group, within each year
subscribed = clean_df["subscribed_term_deposit"] == "yes"
year = clean_df["inferred_campaign_year"]

previous = clean_df["previous_campaign_outcome"].map(
    {"nonexistent": "never contacted", "failure": "earlier failure", "success": "earlier success"})
age_group = pd.cut(clean_df["age"], bins=[0, 25, 35, 55, 120],
                   labels=["under 25", "25-34", "35-54", "55 and over"], right=False)
segment = previous + " / " + clean_df["contact_type"] + " / " + age_group.astype(str)

# Reconcile the labels against the earlier tables
print("Previous outcome:", previous.value_counts().to_dict(),
      "(should be never contacted 35551, earlier failure 4252, earlier success 1373)")
print("Contact type:", clean_df["contact_type"].value_counts().to_dict(),
      "(should be cellular 26135, telephone 15041)")
print("Age group:", age_group.value_counts().sort_index().to_dict(),
      "(should be under 25 1067, 25-34 13684, 35-54 22197, 55 and over 4228)")
print()

frame = pd.DataFrame({"segment": segment, "year": year, "subscribed": subscribed})
grouped = frame.groupby(["segment", "year"])["subscribed"].agg(customers="count", subscribers="sum")
customers = grouped["customers"].unstack("year").fillna(0).astype(int)
subscribers = grouped["subscribers"].unstack("year").fillna(0).astype(int)

print(f"Segments found: {len(customers)} (at most 24)")
print(f"Customers: {customers.values.sum()} (should be 41176), by year: {customers.sum().to_dict()} "
      f"(should be {{2008: 27682, 2009: 11436, 2010: 2058}})")
print(f"Subscribers: {subscribers.values.sum()} (should be 4639), by year: {subscribers.sum().to_dict()} "
      f"(should be {{2008: 1339, 2009: 2227, 2010: 1073}})")
print()

year_rate = subscribers.sum() / customers.sum() * 100        # yes rate of each year
rate = (subscribers / customers * 100).where(customers >= 100)
times_year_rate = (rate / year_rate).round(2)

def pattern(ratios):
    shown = ratios.dropna()
    if len(shown) < 2:
        return "one year only"
    if (shown > 1).all():
        return "above in every year shown"
    if (shown < 1).all():
        return "below in every year shown"
    return "mixed"

result = pd.concat({"customers": customers, "yes_rate_pct": rate.round(1),
                    "times_year_rate": times_year_rate}, axis=1)
result.columns = [f"{name}_{y}" for name, y in result.columns]
result["total_customers"] = customers.sum(axis=1)
result["pattern"] = times_year_rate.apply(pattern, axis=1)

print(f"Yes rate of each year (%): {year_rate.round(1).to_dict()}")
print("Segments (rates blank where under 100 customers), sorted by 2009 times the year's rate:")
result.sort_values("times_year_rate_2009", ascending=False)

Previous outcome: {'never contacted': 35551, 'earlier failure': 4252, 'earlier success': 1373} (should be never contacted 35551, earlier failure 4252, earlier success 1373)
Contact type: {'cellular': 26135, 'telephone': 15041} (should be cellular 26135, telephone 15041)
Age group: {'under 25': 1067, '25-34': 13684, '35-54': 22197, '55 and over': 4228} (should be under 25 1067, 25-34 13684, 35-54 22197, 55 and over 4228)

Segments found: 24 (at most 24)
Customers: 41176 (should be 41176), by year: {2008: 27682, 2009: 11436, 2010: 2058} (should be {2008: 27682, 2009: 11436, 2010: 2058})
Subscribers: 4639 (should be 4639), by year: {2008: 1339, 2009: 2227, 2010: 1073} (should be {2008: 1339, 2009: 2227, 2010: 1073})

Yes rate of each year (%): {2008: 4.8, 2009: 19.5, 2010: 52.1}
Segments (rates blank where under 100 customers), sorted by 2009 times the year's rate:


,customers_2008,customers_2009,customers_2010,yes_rate_pct_2008,yes_rate_pct_2009,yes_rate_pct_2010,times_year_rate_2008,times_year_rate_2009,times_year_rate_2010,total_customers,pattern
segment,,,,,,,,,,,
earlier success / cellular / 55 and over,3,134,139,NaN,66.4,82.0,NaN,3.41,1.57,276,above in every year shown
earlier success / cellular / 25-34,9,208,200,NaN,55.8,72.5,NaN,2.86,1.39,417,above in every year shown
earlier success / cellular / 35-54,20,273,197,NaN,52.0,80.7,NaN,2.67,1.55,490,above in every year shown
never contacted / cellular / 55 and over,1236,752,149,5.7,31.0,47.7,1.19,1.59,0.91,2137,mixed
never contacted / cellular / under 25,225,328,30,7.6,30.2,NaN,1.56,1.55,NaN,583,above in every year shown
earlier failure / cellular / under 25,0,105,51,NaN,22.9,NaN,NaN,1.17,NaN,156,one year only
earlier failure / cellular / 55 and over,52,290,115,NaN,22.8,43.5,NaN,1.17,0.83,457,mixed
never contacted / cellular / 25-34,4440,2676,236,6.3,20.3,56.8,1.31,1.04,1.09,7352,above in every year shown
never contacted / telephone / 55 and over,1175,102,36,4.1,18.6,NaN,0.84,0.96,NaN,1313,below in every year shown


In [25]:
# Q10, step 2: exact counts for each segment, for the findings table
exact = pd.DataFrame({
    "customers": customers.sum(axis=1),
    "subscribed": subscribers.sum(axis=1),
})
exact["yes_rate_pct"] = (exact["subscribed"] / exact["customers"] * 100).round(1)
exact["share_of_customers_pct"] = (exact["customers"] / len(clean_df) * 100).round(1)
exact["share_of_subscriptions_pct"] = (exact["subscribed"] / subscribed.sum() * 100).round(1)
for y in [2008, 2009, 2010]:
    exact[f"subscribed_{y}"] = subscribers[y]
exact["pattern"] = result["pattern"]

print(f"Customers: {exact['customers'].sum()} (should be 41176), "
      f"subscribed: {exact['subscribed'].sum()} (should be 4639)")
print()

# Reconcile against Question 7: totals by previous outcome
by_outcome = exact.groupby(exact.index.str.split(" / ").str[0])[["customers", "subscribed"]].sum()
print("By previous outcome (should be never contacted 35551 / 3140, "
      "earlier failure 4252 / 605, earlier success 1373 / 894):")
print(by_outcome)
print()

# The blocks described in the text
blocks = {
    "three best segments (earlier success, cellular, ages 25-34, 35-54, 55+)": [
        "earlier success / cellular / 25-34", "earlier success / cellular / 35-54",
        "earlier success / cellular / 55 and over"],
    "earlier failure, cellular, ages 25-54": [
        "earlier failure / cellular / 25-34", "earlier failure / cellular / 35-54"],
    "never contacted, telephone, ages 25-54": [
        "never contacted / telephone / 25-34", "never contacted / telephone / 35-54"],
    "measurable in two or more years": list(exact.index[exact["pattern"] != "one year only"]),
    "measurable in one year only": list(exact.index[exact["pattern"] == "one year only"]),
}
block_rows = []
for name, members in blocks.items():
    block = exact.loc[members]
    block_rows.append({
        "block": name,
        "segments": len(members),
        "customers": block["customers"].sum(),
        "subscribed": block["subscribed"].sum(),
        "yes_rate_pct": round(block["subscribed"].sum() / block["customers"].sum() * 100, 1),
        "share_of_customers_pct": round(block["customers"].sum() / len(clean_df) * 100, 1),
        "share_of_subscriptions_pct": round(block["subscribed"].sum() / subscribed.sum() * 100, 1),
    })
print("Blocks (customers should be 1183, 3339, 13106, 40311 and 865):")
print(pd.DataFrame(block_rows).set_index("block"))
print()

# Cellular and telephone among earlier successes
earlier_success = exact[exact.index.str.startswith("earlier success")]
print(f"Earlier successes reached by cellular: "
      f"{earlier_success[earlier_success.index.str.contains('cellular')]['customers'].sum()} (should be 1270), "
      f"by telephone: {earlier_success[earlier_success.index.str.contains('telephone')]['customers'].sum()} (should be 103)")
print()

print("Every segment, sorted as in Cell 3:")
exact.loc[result.sort_values("times_year_rate_2009", ascending=False).index]

Customers: 41176 (should be 41176), subscribed: 4639 (should be 4639)

By previous outcome (should be never contacted 35551 / 3140, earlier failure 4252 / 605, earlier success 1373 / 894):
                 customers  subscribed
segment                               
earlier failure       4252         605
earlier success       1373         894
never contacted      35551        3140

Blocks (customers should be 1183, 3339, 13106, 40311 and 865):
                                                    segments  customers  subscribed  yes_rate_pct  share_of_customers_pct  share_of_subscriptions_pct
block                                                                                                                                                
three best segments (earlier success, cellular,...         3       1183         768          64.9                     2.9                        16.6
earlier failure, cellular, ages 25-54                      2       3339         406          12.2     

,customers,subscribed,yes_rate_pct,share_of_customers_pct,share_of_subscriptions_pct,subscribed_2008,subscribed_2009,subscribed_2010,pattern
segment,,,,,,,,,
earlier success / cellular / 55 and over,276,204,73.9,0.7,4.4,1,89,114,above in every year shown
earlier success / cellular / 25-34,417,263,63.1,1.0,5.7,2,116,145,above in every year shown
earlier success / cellular / 35-54,490,301,61.4,1.2,6.5,0,142,159,above in every year shown
never contacted / cellular / 55 and over,2137,375,17.5,5.2,8.1,71,233,71,mixed
never contacted / cellular / under 25,583,129,22.1,1.4,2.8,17,99,13,above in every year shown
earlier failure / cellular / under 25,156,46,29.5,0.4,1.0,0,24,22,one year only
earlier failure / cellular / 55 and over,457,119,26.0,1.1,2.6,3,66,50,mixed
never contacted / cellular / 25-34,7352,959,13.0,17.9,20.7,281,544,134,above in every year shown
never contacted / telephone / 55 and over,1313,81,6.2,3.2,1.7,48,19,14,below in every year shown
